## Test Day

In [8]:
if False:
    import os
    os.environ["QUANTDATA_CACHE_MODE"] = "off" # turns off global cache mode; good for testing, but do not use in general.

%load_ext autoreload
%autoreload 1

%aimport quantdata
%aimport quantdata.biz_calendar
%aimport quantdata.widb
%aimport quantdata.widb.client
%aimport quantdata.widb.day
%aimport quantdata.widb.arbfree

from datetime import date
from quantdata.widb.day import collect_raw_day, enrich_day, ExpiryFilter, RawDayFilter
from quantdata.widb.arbfree import mk_arb_free_day, MkArbFree
from quantdata.biz_calendar import next_business_day
from cdxcore.verbose import Context
from IPython.display import display

def test():
    ticker = "SPX"
    d      = next_business_day( date(2006, 1, 19), count_start=True )

    raw_day_filter = RawDayFilter()
    expiry_filter = ExpiryFilter()
    verbose = Context.all

    raw_day = collect_raw_day(ticker, d, raw_day_filter=raw_day_filter, verbose=verbose, raise_if_bad=False)
    #display(raw_day)

    enr_day = enrich_day(ticker, d, raw_day, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter, verbose=verbose, raise_if_bad=False, override_cache_mode="off")
    #display(enr_day)

    err = enr_day[enr_day['pure_ask_price'] <= enr_day['pure_bid_price']]
    if len(err) > 0:
        display(err[['expiry','strike','pure_ask_price','pure_bid_price']])
        raise ValueError("Bad data: found options with ask price below or equal to bid price.")
    
    arb_free = MkArbFree()
    arb_free = mk_arb_free_day(ticker, d, enr_day, mk_arb_free=arb_free, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter, verbose=verbose, raise_if_bad=False, override_cache_mode="off")

    display(arb_free)
    return arb_free

_ = test()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


AssertionError: Internal error: synthetic call bid price is negative for expiry_id 200601210. Please contact the authors with an example.